# Cost Optimization in Resilient Microgrid Networks — QCi Dirac-3
### 2026 Global Industry Challenge · Energy Infrastructure Track

**Team QMatrix:** Sharmila L · Temitope Akinsunmade · Abdullahi Tajudeen O. · Joseph Falade · Udochukwu Okorie

---

This notebook reproduces the Phase 3 results for team QMatrix on the ARPA-E GO Challenge 1 grid `Network_03O-10` (793 buses, 904 branches/transformers, 82 active generators, 7,801.5 MW load).

**Quantum contribution — non-convex cubic economic dispatch on Dirac-3.** Thermal generation modelled with the higher-fidelity cubic cost `C(P)=aP³+bP²+cP+d` is non-convex, and convex LP/QP solvers must relax it. Economic dispatch is a natural fit for Dirac-3's continuous entropy-quantum encoding: the objective is a degree-3 polynomial and the power-balance constraint `ΣPᵢ = Demand` is the device `sum_constraint`. We run the dispatch on Dirac-3 hardware through `eqc-models` and benchmark it against a convex QP baseline and a multistart global optimum on the same instance, for two cost models (a dataset-fitted control and a non-convex valve-point model) across ten operating scenarios.

**Classical resilience pipeline.** Spectral partitioning into five microgrids (23 PCC tie-lines), an LP DC-OPF sweep over all 91 N-1 contingencies, a transmission-blackout plus secondary-outage study, and contingency-aware DER siting.

Run the cells in order. The dispatch benchmark issues live Dirac-3 jobs; the ten-scenario study is cached in `doc/stats_dispatch.json` and can be regenerated with `python experiments/dispatch_benchmark.py`.


In [ ]:
# Step 1 — Environment and imports
import os, sys, subprocess, time, json
from pathlib import Path
import importlib.util

# Install any missing dependencies (e.g. on a fresh qBraid kernel).
required = ['numpy', 'scipy', 'networkx', 'matplotlib', 'pandapower', 'sklearn', 'eqc_models', 'dotenv']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements.txt'])

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.grid_graph import GridGraphModel
from models.classical_solver import ClassicalMicrogridSolver
from models.cost_models import (
    dataset_cubic_generators, literature_nonconvex_generators, characterize_convexity)
from models.dispatch_dirac3 import NonConvexDispatch

print('Modules loaded.')


In [ ]:
# Step 2 — Load the ARPA-E GO Challenge grid (Network_03O-10)
raw_f = ROOT / 'Original_Dataset_Offline_Edition_1' / 'Network_03O-10' / 'scenario_1' / 'case.raw'
rop_f = ROOT / 'Original_Dataset_Offline_Edition_1' / 'Network_03O-10' / 'case.rop'
con_f = ROOT / 'Original_Dataset_Offline_Edition_1' / 'Network_03O-10' / 'scenario_1' / 'case.con'

grid = GridGraphModel(raw_f, rop_f, con_f)
solver = ClassicalMicrogridSolver(grid)

n_buses = grid.graph.number_of_nodes()
n_edges = grid.graph.number_of_edges()
total_load = sum(grid.graph.nodes[b].get('p_load', 0.0) for b in grid.graph.nodes())
total_crit = sum(grid.graph.nodes[b].get('p_load', 0.0) for b in grid.graph.nodes()
                 if grid.graph.nodes[b].get('is_critical', False))
pcc_edges = [(u, v) for u, v, d in grid.graph.edges(data=True) if d.get('is_pcc', False)]

print(f'Network: {n_buses} buses, {n_edges} branches/transformers, {len(grid.generators)} active generators')
print(f'Load:    {total_load:,.1f} MW total, {total_crit:,.1f} MW critical')


In [ ]:
# Step 3 — Spectral microgrid partition and DER upgrade plan
clusters = grid.identify_microgrids_spectral(n_clusters=5)
upgrades = grid.compute_microgrid_upgrade_plan(clusters)
cost_map = grid.build_generator_cost_map()
base_cost = grid.evaluate_generation_cost(cost_map)
total_upgrade_cost = sum(u['upgrade_cost_usd'] for u in upgrades)

print('Microgrid clusters and N-1 upgrade sizing:')
for u in upgrades:
    print(f'  Cluster {u["cluster_id"]}: {u["num_buses"]:>3} buses, load {u["total_load_mw"]:>7.1f} MW, '
          f'gen {u["existing_gen_mw"]:>7.1f} MW, N-1 deficit {u["gen_deficit_mw"]:>6.1f} MW, '
          f'upgrade ${u["upgrade_cost_usd"]:,.0f}')
print(f'Base-case generation cost: ${base_cost:,.0f}/h   Total upgrade cost: ${total_upgrade_cost:,.0f}')


In [ ]:
# Step 4 — Classical N-1 contingency sweep (LP DC-OPF)
con_list = grid.con_data
RESTORATION_HOURS = 4.0  # assumed restoration time for the critical-downtime metric
sweep_res = []
for c in con_list:
    if c['type'] == 'branch_out':
        r = solver.solve_dc_opf(tripped_branch=c)
    else:
        r = solver.solve_dc_opf(tripped_gen=c)
    shed = r['unserved_load_mw']
    dt = (min(shed, total_crit) / total_crit) * RESTORATION_HOURS if total_crit > 0 else 0.0
    sweep_res.append({'name': c['name'], 'type': c['type'], 'shed_mw': shed, 'downtime_h': dt})

avg_shed = np.mean([s['shed_mw'] for s in sweep_res])
max_shed = max(s['shed_mw'] for s in sweep_res)
max_dt = max(s['downtime_h'] for s in sweep_res)

print(f'Evaluated {len(con_list)} N-1 contingencies (62 branch, 29 generator outages).')
print(f'Intact grid: mean unserved {avg_shed:.2f} MW (max {max_shed:.2f} MW), '
      f'max critical downtime {max_dt:.2f} h')


In [ ]:
# Step 5 — Non-convex cubic economic dispatch on QCi Dirac-3
#
# Dispatch minimises total cost s.t. power balance and generator limits:
#   min sum_i C_i(P_i)   s.t.   sum_i P_i = D,   Pmin_i <= P_i <= Pmax_i
# With the cubic cost C_i(P) = a P^3 + b P^2 + c P + d the objective is a
# degree-3 polynomial and the balance constraint sum_i P_i = D is the Dirac-3
# sum_constraint. We solve on hardware and compare against a convex QP and a
# multistart global optimum on the same instance.

pcc_lines = [(u, v) for u, v, d in grid.graph.edges(data=True) if d.get('is_pcc', False)]
pcc_all = pcc_lines  # used by the resilience analysis (Step 7)

target_cid = max(clusters, key=lambda k: len(clusters[k]))
gensB = literature_nonconvex_generators(grid, clusters[target_cid], ripple=0.9)
gensB.sort(key=lambda g: g.p_max, reverse=True)
gensB = gensB[:10]
dispatch = NonConvexDispatch(gensB)

# Verify non-convexity: each unit's marginal-cost minimum falls inside its
# operating range (the valve-point operating point).
conv_info = [characterize_convexity(g) for g in gensB]
n_nonconvex = sum(c['is_nonconvex'] for c in conv_info)
demand = float(sum(grid.graph.nodes[b].get('p_load', 0.0) for b in clusters[target_cid]))
band = (dispatch.p_min.sum(), dispatch.p_max.sum())
demand = float(np.clip(demand, band[0] + 1, band[1] - 1))

print(f'Microgrid cluster {target_cid}: {len(gensB)} units, {n_nonconvex} non-convex; '
      f'{len(pcc_all)} PCC tie-lines')
print(f'Feasible generation band [{band[0]:.0f}, {band[1]:.0f}] MW; dispatch demand {demand:.0f} MW')
ex = conv_info[0]
print(f'  unit at bus {ex["bus"]}: marginal-cost minimum at {ex["marginal_min_at_MW"]} MW '
      f'within [{ex["p_min"]:.0f}, {ex["p_max"]:.0f}] MW')

conv = dispatch.solve_convex_qp(demand)
glob = dispatch.solve_global_reference(demand, n_starts=80)
dirac = dispatch.solve_dirac3(demand, relaxation_schedule=3, num_samples=20)

gap = lambda x: (x - glob.total_cost) / glob.total_cost * 100
saving = conv.total_cost - dirac.total_cost
print('\nNon-convex dispatch, single scenario (real Dirac-3 hardware):')
print(f'  Convex QP baseline   {conv.total_cost:>12,.1f} $/h   (+{gap(conv.total_cost):.2f}% vs global)')
print(f'  Dirac-3 EQC          {dirac.total_cost:>12,.1f} $/h   (+{gap(dirac.total_cost):.2f}% vs global)')
print(f'  Global optimum       {glob.total_cost:>12,.1f} $/h')
print(f'  Saving vs convex QP  {saving:>12,.1f} $/h   ({saving/conv.total_cost*100:.2f}%)')
print(f'  Power-balance error  {dirac.balance_error:.3f} MW   feasible={dirac.feasible}   '
      f'runtime={dirac.runtime_sec:.1f} s')


In [ ]:
# Step 6 — Ten-scenario validation and dataset control
#
# Experiment A: dataset-fitted cubic (near-convex) — control case.
# Experiment B: non-convex valve-point cubic across the ten dataset scenarios.
# Results are produced by experiments/dispatch_benchmark.py and cached to
# doc/stats_dispatch.json (regenerate: python experiments/dispatch_benchmark.py).
stats_file = ROOT / 'doc' / 'stats_dispatch.json'
if not stats_file.exists():
    subprocess.check_call([sys.executable, 'experiments/dispatch_benchmark.py'])
S = json.loads(stats_file.read_text())

A = S['experiment_A_dataset_control']
print('Experiment A — dataset-fitted cubic (control):')
print(f'  {S["experiment_A_nonconvex_units"]}/{S["n_dispatch_units"]} units non-convex; '
      f'convex QP {A["convex_qp_cost"]:,.0f} $/h, Dirac-3 {A["dirac3_cost"]:,.0f} $/h, '
      f'global {A["global_cost"]:,.0f} $/h (Dirac-3 within {A["dirac3_gap_pct"]:.2f}% of optimum).\n')

sweep = S['experiment_B_sweep']
sm = S['summary']
print('Experiment B — non-convex cubic across ten scenarios:')
print(f'{"scn":>3} | {"demand":>8} | {"convex QP":>11} | {"Dirac-3":>11} | {"global":>11} | {"saving":>8}')
for r in sweep:
    print(f'{r["scenario"]:>3} | {r["demand_mw"]:>7.0f}M | {r["convex_qp_cost"]:>11,.0f} | '
          f'{r["dirac3_cost"]:>11,.0f} | {r["global_cost"]:>11,.0f} | {r["dirac3_saving_vs_convex_pct"]:>7.2f}%')
print('-' * 66)
print(f'Mean saving vs convex QP: {sm["dirac3_mean_saving_vs_convex_pct"]:.2f}%   '
      f'Mean gap to global: Dirac-3 {sm["dirac3_mean_gap_vs_global_pct"]:.2f}% vs '
      f'convex QP {sm["convex_qp_mean_gap_vs_global_pct"]:.2f}%')
print(f'Feasibility: all {sm["scenarios_evaluated"]} scenarios feasible, '
      f'max power-balance error {sm["dirac3_max_balance_error_mw"]} MW')

scn = [r['scenario'] for r in sweep]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
w = 0.27
ax[0].bar([s - w for s in scn], [r['convex_qp_cost'] for r in sweep], w, label='Convex QP', color='#b3452f')
ax[0].bar(scn, [r['dirac3_cost'] for r in sweep], w, label='Dirac-3 EQC', color='#2f6fb3')
ax[0].bar([s + w for s in scn], [r['global_cost'] for r in sweep], w, label='Global optimum', color='#3a8a3a')
ax[0].set_xlabel('Scenario'); ax[0].set_ylabel('Dispatch cost ($/h)')
ax[0].set_title('Dispatch cost by scenario'); ax[0].legend(frameon=False); ax[0].grid(alpha=.3, axis='y')
ax[1].plot(scn, [r['convex_qp_gap_pct'] for r in sweep], 'o-', color='#b3452f', label='Convex QP')
ax[1].plot(scn, [r['dirac3_gap_pct'] for r in sweep], 's-', color='#2f6fb3', label='Dirac-3 EQC')
ax[1].axhline(0, color='#3a8a3a', lw=1)
ax[1].set_xlabel('Scenario'); ax[1].set_ylabel('Optimality gap vs global (%)')
ax[1].set_title('Optimality gap'); ax[1].legend(frameon=False); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# Step 7 — Transmission blackout and per-island resilience
#
# Loss of all 23 PCC tie-lines forces the grid to operate as five islands. We
# evaluate each island with DC-OPF, first at base load and then under a
# secondary N-1 generator outage, with and without the sited DER upgrades.
extra_gen = {}
for upg in upgrades:
    cid = upg['cluster_id']
    added_mw = upg['add_microturbine_mw'] + upg['add_bess_mw']
    if added_mw > 0:
        host_bus = max(clusters[cid], key=lambda b: grid.graph.nodes[b].get('p_load', 0.0))
        extra_gen[host_bus] = added_mw

bo_base = solver.solve_dc_opf_with_tripped_pccs(pcc_all)

c3_gens = [(gk, g) for gk, g in grid.generators.items() if g['bus'] in clusters[3] and g['stat'] == 1]
worst_gen = max(c3_gens, key=lambda x: x[1]['pt']) if c3_gens else None
trip_gen = {'bus': worst_gen[1]['bus'], 'unit_id': worst_gen[0][1]} if worst_gen else None

bo_n1 = solver.solve_dc_opf_with_tripped_pccs(pcc_all, tripped_gen=trip_gen)
bo_n1_der = solver.solve_dc_opf_with_tripped_pccs(pcc_all, extra_gen_mw=extra_gen, tripped_gen=trip_gen)

print('Transmission blackout (five-island operation):')
print(f'  Base load:                    {bo_base["unserved_load_mw"]:6.2f} MW unserved '
      f'({bo_base["num_islands"]} islands)')
print(f'  Secondary N-1, no upgrade:    {bo_n1["unserved_load_mw"]:6.2f} MW unserved '
      f'({bo_n1["unserved_pct"]:.2f}%)')
print(f'  Secondary N-1, with DER:      {bo_n1_der["unserved_load_mw"]:6.2f} MW unserved '
      f'(${total_upgrade_cost/1e6:.1f}M upgrade)')


In [ ]:
# Step 8 — Dirac-3 encoding of the cubic dispatch
#
# Using the shift P_i = Pmin_i + y_i (y_i >= 0), lower limits hold by
# construction and the device sum_constraint enforces sum_i y_i = D - sum Pmin.
# Expanding C(Pmin + y) gives a degree-3 polynomial in y, submitted to
# eqc-models as (coefficients, indices). Index rows are 1-indexed with 0 padding:
# [0,0,v] linear, [0,v,v] quadratic, [v,v,v] cubic.
coeffs, indices = dispatch.build_polynomial()
reduced = demand - dispatch.p_min.sum()
print(f'Polynomial operator: {len(coeffs)} terms over {dispatch.n} continuous variables (degree 3).')
print(f'sum_constraint = D - sum(Pmin) = {demand:.0f} - {dispatch.p_min.sum():.0f} = {reduced:.0f} MW')
print('Terms for generator 1 (variable index 1):')
for cf, ix in zip(coeffs[:3], indices[:3]):
    degree = {1: 'linear', 2: 'quadratic', 3: 'cubic'}[int(np.count_nonzero(ix))]
    print(f'  {degree:<10} coefficient {cf:+.5g}   indices {list(ix)}')
print(f'Device footprint: {dispatch.n} continuous modes, full connectivity, '
      'relaxation_schedule=3, num_samples=20.')


In [ ]:
# Step 9 — Results summary
print('Dataset            ', f'ARPA-E GO Challenge 1, {raw_f.parent.parent.name}')
print('Network            ', f'{n_buses} buses, {n_edges} branches/transformers, '
      f'{len(grid.generators)} active generators')
print('Load               ', f'{total_load:,.1f} MW total, {total_crit:,.1f} MW critical '
      f'(>= {grid.CRITICAL_LOAD_THRESHOLD_MW:.0f} MW)')
print('Microgrids         ', f'{len(clusters)} clusters, {len(pcc_all)} PCC tie-lines')
print('N-1 sweep          ', f'{len(con_list)} contingencies, {avg_shed:.2f} MW mean shed on intact grid')
print('Blackout + N-1     ', f'{bo_n1["unserved_load_mw"]:.2f} MW unserved, reduced to '
      f'{bo_n1_der["unserved_load_mw"]:.2f} MW with ${total_upgrade_cost/1e6:.1f}M DER')
print('Dispatch (Dirac-3) ', f'cluster {target_cid}, {len(gensB)} units ({n_nonconvex} non-convex); '
      f'mean saving {S["summary"]["dirac3_mean_saving_vs_convex_pct"]:.2f}% vs convex QP over 10 scenarios')
print('                   ', f'mean optimality gap {S["summary"]["dirac3_mean_gap_vs_global_pct"]:.2f}% '
      f'(convex QP {S["summary"]["convex_qp_mean_gap_vs_global_pct"]:.2f}%); '
      f'control gap {A["dirac3_gap_pct"]:.2f}%')
